# 烂片网络

In [1]:
import pandas as pd
import numpy as np
import networkx as nx
import scipy
import matplotlib.pyplot as plt
plt.rcParams['font.sans-serif'] = ['Heiti TC']
plt.rcParams['axes.unicode_minus'] = False
from networkx.algorithms import bipartite
import json

In [2]:
import sys
sys.path.append("..")

# 数据处理

In [3]:
df_raw = pd.read_csv("../data/movie_data_rated.csv")
df_raw['k_cast_id'] = df_raw['k_cast_id'].apply(lambda x: str(x))
df_raw

,cast_id,cast_name,gender,movie_id,k_role,is_main_cast,index,id,k_title,k_type,...,rating_num,rating_people,k_rating_people,rating_index,k_movie_id,k_cast_id,uid,portray,movie_id_m,cast_role_agg
0,1017182,陈燕燕,女,10771202,演员,是,1,10771202,深闺疑云,电影,...,7.8,92,92.0,27.0,21542453,2034413,54,赵兰,m21542453,演员
1,1043845,刘志荣,男,10581318,演员,是,11,10581318,四大家族之龙虎兄弟,电影,...,7.8,2290,2290.0,134.0,21162685,2087739,75,NaN,m21162685,导演/演员
2,1050373,茅瑛,女,10573512,演员,是,1,10573512,鬼娘子,电影,...,5.8,235,235.0,37.0,21147073,2100795,77,NaN,m21147073,演员
3,1124360,利智,女,10581318,演员,是,3,10581318,四大家族之龙虎兄弟,电影,...,7.8,2290,2290.0,134.0,21162685,2248769,92,NaN,m21162685,演员
4,1314471,王挺,男,10531190,演员,是,1,10531190,惊天动地,电视剧,...,8.5,326,326.0,53.0,21062429,2628991,146,NaN,m21062429,导演/演员
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
256797,1354509,林泳淘,女,26984991,演员,否,999,26984991,婚姻合伙人,电视剧,...,5.8,1259,1259.0,85.0,53970031,2709067,604125,邓秀玉,m53970031,演员
256798,27496534,刘芊蒂,女,2337597,演员,是,2,2337597,操行零分,电影,...,7.0,211,211.0,38.0,4675243,54993117,604128,NaN,m4675243,演员
256799,27565771,董亚春,女,2270516,导演,是,2,2270516,八月一日,电影,...,6.4,1123,1123.0,85.0,4541081,55131591,604130,NaN,m4541081,导演
256800,35415094,张可,女,7053738,演员,否,999,7053738,樱桃,电视剧,...,5.7,2003,2003.0,107.0,14107525,70830237,604135,NaN,m14107525,演员


## 电影数据

In [4]:
df_movie_raw = df_raw.loc[(df_raw['k_type'] == '电影').copy()]
df_movie_raw

,cast_id,cast_name,gender,movie_id,k_role,is_main_cast,index,id,k_title,k_type,...,rating_num,rating_people,k_rating_people,rating_index,k_movie_id,k_cast_id,uid,portray,movie_id_m,cast_role_agg
0,1017182,陈燕燕,女,10771202,演员,是,1,10771202,深闺疑云,电影,...,7.8,92,92.0,27.0,21542453,2034413,54,赵兰,m21542453,演员
1,1043845,刘志荣,男,10581318,演员,是,11,10581318,四大家族之龙虎兄弟,电影,...,7.8,2290,2290.0,134.0,21162685,2087739,75,NaN,m21162685,导演/演员
2,1050373,茅瑛,女,10573512,演员,是,1,10573512,鬼娘子,电影,...,5.8,235,235.0,37.0,21147073,2100795,77,NaN,m21147073,演员
3,1124360,利智,女,10581318,演员,是,3,10581318,四大家族之龙虎兄弟,电影,...,7.8,2290,2290.0,134.0,21162685,2248769,92,NaN,m21162685,演员
6,1315281,卢庆辉,男,10490155,演员,是,1,10490155,衰鬼抓狂,电影,...,5.8,121,121.0,26.0,20980359,2630611,156,NaN,m20980359,演员
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
256792,27517568,林迪安,男,1300498,演员,否,999,1300498,百变星君,电影,...,7.7,246162,246162.0,1377.0,2601045,55035185,604118,穿斑马服君,m2601045,导演/演员
256793,27541395,韩鹏翼,男,24695588,演员,是,2,24695588,逆袭,电影,...,5.1,674,674.0,59.0,49391225,55082839,604119,NaN,m49391225,演员
256798,27496534,刘芊蒂,女,2337597,演员,是,2,2337597,操行零分,电影,...,7.0,211,211.0,38.0,4675243,54993117,604128,NaN,m4675243,演员
256799,27565771,董亚春,女,2270516,导演,是,2,2270516,八月一日,电影,...,6.4,1123,1123.0,85.0,4541081,55131591,604130,NaN,m4541081,导演


In [5]:
df_movie = df_movie_raw[['movie_id_m', 'k_title',
                   'rating_num', 'k_rating_people']].drop_duplicates().reset_index(drop=True)
df_movie

,movie_id_m,k_title,rating_num,k_rating_people
0,m21542453,深闺疑云,7.8,92.0
1,m21162685,四大家族之龙虎兄弟,7.8,2290.0
2,m21147073,鬼娘子,5.8,235.0
3,m20980359,衰鬼抓狂,5.8,121.0
4,m21480521,冬去春来,5.7,42.0
...,...,...,...,...
11059,m53335019,武则天降妖记,2.2,378.0
11060,m9414485,黑蛋，快跑,7.0,69.0
11061,m53623223,大护法,7.8,326831.0
11062,m70564915,大雨,6.3,27389.0


In [6]:
# 评分人数>=10万
df_movie_rate_num_10w = df_movie[df_movie['k_rating_people'] >= 100000]
df_movie_rate_num_1w = df_movie[df_movie['k_rating_people'] >= 10000]

## 电视剧数据

In [7]:
df_tv_raw = df_raw.loc[(df_raw['k_type'] == '电视剧').copy()]
df_tv_raw

,cast_id,cast_name,gender,movie_id,k_role,is_main_cast,index,id,k_title,k_type,...,rating_num,rating_people,k_rating_people,rating_index,k_movie_id,k_cast_id,uid,portray,movie_id_m,cast_role_agg
4,1314471,王挺,男,10531190,演员,是,1,10531190,惊天动地,电视剧,...,8.5,326,326.0,53.0,21062429,2628991,146,NaN,m21062429,导演/演员
5,1315195,孟智超,男,10441617,演员,是,3,10441617,逼子成龙,电视剧,...,6.7,83,83.0,24.0,20883283,2630439,152,NaN,m20883283,演员
7,1315174,詹秉熙,男,10465144,演员,是,17,10465144,火速救兵2,电视剧,...,7.4,58,58.0,21.0,20930337,2630397,161,NaN,m20930337,演员
11,1316563,李承炫,男,10535482,演员,是,4,10535482,宝贝妈妈宝贝女,电视剧,...,6.7,974,974.0,81.0,21071013,2633175,181,NaN,m21071013,演员
16,27547196,谭凯,男,10569157,演员,是,5,10569157,暗花,电视剧,...,6.0,119,119.0,27.0,21138363,55094441,205,NaN,m21138363,演员
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
256794,27567475,张羽,男,5318674,演员,否,999,5318674,世间道,电视剧,...,7.9,113,113.0,30.0,10637397,55134999,604120,洪宝强,m10637397,导演/演员
256795,27583497,王侃伟,男,27011418,演员,是,55,27011418,秋蝉,电视剧,...,5.7,55105,55105.0,560.0,54022885,55167043,604121,NaN,m54022885,演员
256796,36085879,欧阳振峰,男,21348054,演员,否,999,21348054,上阵父子兵,电视剧,...,6.4,557,557.0,60.0,42696157,72171807,604123,NaN,m42696157,演员
256797,1354509,林泳淘,女,26984991,演员,否,999,26984991,婚姻合伙人,电视剧,...,5.8,1259,1259.0,85.0,53970031,2709067,604125,邓秀玉,m53970031,演员


In [8]:
df_tv = df_tv_raw[['movie_id_m', 'k_title',
                   'rating_num', 'k_rating_people']].drop_duplicates().reset_index(drop=True)
df_tv

,movie_id_m,k_title,rating_num,k_rating_people
0,m21062429,惊天动地,8.5,326.0
1,m20883283,逼子成龙,6.7,83.0
2,m20930337,火速救兵2,7.4,58.0
3,m21071013,宝贝妈妈宝贝女,6.7,974.0
4,m21138363,暗花,6.0,119.0
...,...,...,...,...
8029,m52674745,快乐星猫 第四季,8.4,145.0
8030,m53539177,味道天津 第二季,7.1,88.0
8031,m49741021,好味·广州,6.3,231.0
8032,m9717585,怦然心动,7.9,205.0


In [82]:
# 评分人数>=10万
df_tv_rate_num_10w = df_tv[df_tv['k_rating_people'] >= 100000]
df_tv_rate_num_1w = df_tv[df_tv['k_rating_people'] >= 10000]
df_tv_rate_num_5w = df_tv[df_tv['k_rating_people'] >= 50000]
df_tv_rate_num_2w = df_tv[df_tv['k_rating_people'] >= 20000]

# 方法

In [10]:
# 生成图
def get_G(df):
    G = nx.from_pandas_edgelist(
        df,
        source='k_cast_id',
        target='movie_id_m',
        edge_attr=True,
        create_using=nx.Graph()
    )
    return G

In [11]:
# 最大连通子图
def get_largest_connected_component(G):
    if nx.is_connected(G):
        return G
    else:
        largest_cc = max(nx.connected_components(G), key=len)
        return G.subgraph(largest_cc).copy()

In [12]:
# 二分图，投影到电影节点
def get_movie_bip(G, cast_ids):
    return bipartite.projected_graph(G, G.nodes - cast_ids)

In [13]:
# 绘图
def draw_movie_net(G, k=None):
    pos = nx.spring_layout(G, k=k, seed=42)

    plt.figure(figsize=(10, 10))
    nx.draw_networkx(
        G,
        pos=pos,
        with_labels=False,
        node_size=1,
        edge_color="gainsboro",
        alpha=0.4,
    )
    plt.title("Movie Network")
    plt.axis('off')
    plt.show()
    print(G.number_of_nodes(), G.number_of_edges())

In [ ]:
# 按评分提取数据
def get_lowest_rated_movie_casts(df, work_type="电影", ascending=True, top_n=100):
    df_type = df.loc[(df['k_type'] == work_type).copy()]
    df_type_simple = df_type[['movie_id_m', 'k_title',
                   'rating_num', 'k_rating_people']].drop_duplicates().reset_index(drop=True)
    df_type_n = df_type_simple.sort_values(by='rating_num', ascending=ascending).head(top_n)
    df_n = df_type_n[['movie_id_m']].merge(df,
                                            left_on='movie_id_m',
                                            right_on='movie_id_m',
                                            how='left')
    return df_n

In [100]:
# 作品筛选
def works_filter(df, work_type="电影", sort_by='rating_num', ascending=True, top_n=100, rating_people_min=0):
    df_type = df.loc[(df['k_type'] == work_type).copy()]
    df_type_simple = df_type[['movie_id_m', 'k_title',
                   'rating_num', 'k_rating_people']].drop_duplicates().reset_index(drop=True)
    df_type_n = df_type_simple[df_type_simple['k_rating_people'] >= rating_people_min].sort_values(by=sort_by, ascending=ascending).head(top_n)
    df_n = df_type_n[['movie_id_m']].merge(df_type,
                                            left_on='movie_id_m',
                                            right_on='movie_id_m',
                                            how='left')
    return df_n

In [15]:
# 度数为1的影人节点
def degree_1_cast_ids(G):
    degree_1_nodes = [node for node, degree in G.degree() if degree == 1]
    nodes_to_remove = [i for i in degree_1_nodes if 'm' not in str(i)]
    return nodes_to_remove

In [60]:
# 生成图表数据
def get_chart_data(G, df, sub_pos=False, work_type="电影"):
    nodes = G.nodes()
    n = len(nodes)
    k = None if n < 500 else 1.5
    pos = nx.kamada_kawai_layout(G) if sub_pos else nx.spring_layout(G, k=k, seed=42)
    # pos= nx.kamada_kawai_layout(G)
    # pos = nx.forceatlas2_layout(G)
    degrees = dict(G.degree())
    cast_degrees = {k: v for k, v in degrees.items() if 'm' not in str(k)}
    # 将cast_degrees归一到2-7的范围
    cast_degree_values = np.array(list(cast_degrees.values()))
    min_degree = cast_degree_values.min()
    max_degree = cast_degree_values.max()
    for key in cast_degrees:
        cast_degrees[key] = 2 + 5 * (cast_degrees[key] -
                                     min_degree) / (max_degree - min_degree) 
    # 二分图-电影
    movie_nodes = [n for n in nodes if 'm' in str(n)]
    G_movie = bipartite.projected_graph(G, movie_nodes)
    movie_degrees = dict(G_movie.degree())
    # 将movie_degrees归一到2-9的范围
    movie_degree_values = np.array(list(movie_degrees.values()))
    min_degree = movie_degree_values.min()
    max_degree = movie_degree_values.max()
    for key in movie_degrees:
        movie_degrees[key] = 2 + 7 * (movie_degrees[key] -
                                      min_degree) / (max_degree - min_degree)
    # 节点属性
    nodes_data = []
    movie_num = 0
    for node in nodes:
        node_pos = pos[node]
        node_data = {}
        if 'm' in str(node):
            node_data['category'] = work_type
            node_data['title'] = df[df['movie_id_m'] ==
                                    node]['k_title'].values[0]
            node_data['degree'] = movie_degrees.get(node, 0)
            node_data['order'] = 0
            movie_num += 1
        else:
            node_data['category'] = '导演/演员'
            node_data['title'] = df[df['k_cast_id'] ==
                                    node]['cast_name'].values[0]
            node_data['degree'] = cast_degrees.get(node, 0)
            node_data['order'] = 1
        node_data['id'] = str(node)
        node_data['x'] = round(float(node_pos[0]), 6)
        node_data['y'] = round(float(node_pos[1]), 6)
        nodes_data.append(node_data)
    # 边属性
    edges = G.edges()
    edges_data = [{
        'source': str(s),
        'target': str(t),
        'weight': 1
    } for s, t in edges]

    cast_num = len(nodes) - movie_num
    subtitle = f"{movie_num}部{work_type}, {cast_num}位导演与主要演员"
    net_data = {
        'nodes': nodes_data,
        'edges': edges_data,
        'categories': [{
            'name': work_type,
            'order': 0
        }, {
            'name': '导演/演员',
            'order': 1
        }],
        'subtitle': subtitle,
    }
    return net_data

In [61]:
# 生成电影二分图图表数据
def get_movie_bip_chart_data(G, df):
    nodes = G.nodes()
    # 二分图-电影
    movie_nodes = [n for n in nodes if 'm' in str(n)]
    G_movie = bipartite.weighted_projected_graph(G, movie_nodes)
    movie_degrees = dict(G_movie.degree())
    # 将movie_degrees归一到2-9的范围
    movie_degree_values = np.array(list(movie_degrees.values()))
    min_degree = movie_degree_values.min()
    max_degree = movie_degree_values.max()
    for key in movie_degrees:
        movie_degrees[key] = 2 + 7 * (movie_degrees[key] -
                                      min_degree) / (max_degree - min_degree)
    pos = nx.spring_layout(G_movie, k=3, scale=1, seed=412)
    nodes_data = []
    for node in movie_nodes:
        node_data = {}
        node_pos = pos[node]
        node_data['category'] = '电影'
        node_data['title'] = df[df['movie_id_m'] == node]['k_title'].values[0]
        node_data['degree'] = round(float(movie_degrees.get(node, 0)), 2)
        node_data['order'] = 0
        node_data['id'] = str(node)
        node_data['x'] = round(float(node_pos[0]), 6)
        node_data['y'] = round(float(node_pos[1]), 6)
        nodes_data.append(node_data)
    edges = G_movie.edges(data=True)
    edges_data = [{
        'source': str(s),
        'target': str(t),
        'weight': d['weight']
    } for s, t, d in edges]
    movie_num = len(movie_nodes)
    subtitle = f"{movie_num}部电影的关联关系"
    net_data = {
        'nodes': nodes_data,
        'edges': edges_data,
        'categories': [{
            'name': '电影',
            'order': 0
        }],
        'subtitle': subtitle,
    }
    return net_data

In [62]:
def get_chart_data_all(df, title_prefix, work_type="电影"):
    # 整图
    G_n = get_G(df)
    # 最大子图
    G_n_largest = get_largest_connected_component(G_n)
    movie_bip_data = get_movie_bip_chart_data(G_n_largest, df)
    # 去除度为1影人节点
    G_n_node_to_remove = degree_1_cast_ids(G_n)
    G_n_d2 = G_n.copy()
    G_n_d2.remove_nodes_from(G_n_node_to_remove)
    # 最大子图去除度为1影人节点
    G_n_largest_node_to_remove = degree_1_cast_ids(G_n_largest)
    G_n_d2_largest = G_n_largest.copy()
    G_n_d2_largest.remove_nodes_from(G_n_largest_node_to_remove)

    # 生成图表数据
    net_data = get_chart_data(G_n, df, work_type=work_type)
    net_data['title'] = title_prefix + "-全量数据"
    net_data_largest = get_chart_data(G_n_largest, df, work_type=work_type)
    net_data_largest['title'] = title_prefix + "-最大连通子图"
    net_data_d2 = get_chart_data(G_n_d2, df, sub_pos=True, work_type=work_type)
    net_data_d2['title'] = title_prefix + "-去除度为1的影人节点"
    net_data_d2_largest = get_chart_data(G_n_d2_largest, df, sub_pos=True, work_type=work_type)
    net_data_d2_largest['title'] = title_prefix + "-最大连通子图去除度为1的影人节点"
    net_data = {
        "net_data": net_data,
        "net_data_largest": net_data_largest,
        "net_data_d2": net_data_d2,
        "net_data_d2_largest": net_data_d2_largest,
        "movie_bip_data": movie_bip_data
    }
    return net_data

# 电影

## TOP +N
* 评分人数超过10万人

In [19]:
top_n = 200
df_movie_top_n = get_lowest_rated_movie_casts(df_movie_rate_num_10w,
                                              df_movie_raw,
                                              ascending=False,
                                              top_n=top_n)
df_movie_top_n

,movie_id_m,cast_id,cast_name,gender,movie_id,k_role,is_main_cast,index,id,k_title,...,is_rating,rating_num,rating_people,k_rating_people,rating_index,k_movie_id,k_cast_id,uid,portray,cast_role_agg
0,m2583141,1003494,张国荣,男,1291546,演员,是,1,1291546,霸王别姬,...,有评分,9.6,1783862,1783862.0,4138.0,2583141,2007037,389,程蝶衣(小豆子,导演/演员
1,m2583141,27226213,蒋雯丽,女,1291546,演员,是,6,1291546,霸王别姬,...,有评分,9.6,1783862,1783862.0,4138.0,2583141,54452475,1784,小豆子生母,导演/演员
2,m2583141,27256028,张丰毅,男,1291546,演员,是,2,1291546,霸王别姬,...,有评分,9.6,1783862,1783862.0,4138.0,2583141,54512105,3236,段小楼(小石头,演员
3,m2583141,27206595,葛优,男,1291546,演员,是,4,1291546,霸王别姬,...,有评分,9.6,1783862,1783862.0,4138.0,2583141,54413239,3384,袁世卿(袁四爷,演员
4,m2583141,1275231,李春,男,1291546,演员,是,14,1291546,霸王别姬,...,有评分,9.6,1783862,1783862.0,4138.0,2583141,2550511,8887,少年小四,演员
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3985,m43883657,1458583,李彩霞,女,21941804,演员,是,8,21941804,白日焰火,...,有评分,7.6,362165,362165.0,1659.0,43883657,2917215,454389,NaN,演员
3986,m43883657,35422753,王英涛,女,21941804,演员,否,999,21941804,白日焰火,...,有评分,7.6,362165,362165.0,1659.0,43883657,70845555,460080,居委会吕主任,演员
3987,m43883657,27480917,王景春,男,21941804,演员,是,4,21941804,白日焰火,...,有评分,7.6,362165,362165.0,1659.0,43883657,54961883,462373,荣荣,演员
3988,m43883657,1375912,彭龙,男,21941804,演员,是,12,21941804,白日焰火,...,有评分,7.6,362165,362165.0,1659.0,43883657,2751873,533987,便衣,导演/演员


In [20]:
movie_net_data_top = get_chart_data_all(df_movie_top_n, title_prefix="高分电影关系网络")
# 保存处理后的数据
with open('json_data/movie_net_data_highest.json', 'w', encoding='utf-8') as f:
    json.dump(movie_net_data_top, f, ensure_ascii=False, indent=4)

## TOP -N

In [21]:
top_n_min = 250
df_movie_top_n_min = get_lowest_rated_movie_casts(df_movie,
                                              df_movie_raw,
                                              ascending=True,
                                              top_n=top_n_min)
df_movie_top_n_min

,movie_id_m,cast_id,cast_name,gender,movie_id,k_role,is_main_cast,index,id,k_title,...,is_rating,rating_num,rating_people,k_rating_people,rating_index,k_movie_id,k_cast_id,uid,portray,cast_role_agg
0,m51454521,27494382,刘萌萌,女,25727236,演员,是,5,25727236,201413,...,有评分,2.1,1413,1413.0,54.0,51454521,54988813,31845,NaN,演员
1,m51454521,27483544,夏一瑶,女,25727236,演员,是,6,25727236,201413,...,有评分,2.1,1413,1413.0,54.0,51454521,54967137,42136,NaN,演员
2,m51454521,27573296,方品乔,女,25727236,演员,否,999,25727236,201413,...,有评分,2.1,1413,1413.0,54.0,51454521,55146641,73212,NaN,演员
3,m51454521,27551274,彭波,男,25727236,演员,是,8,25727236,201413,...,有评分,2.1,1413,1413.0,54.0,51454521,55102597,182798,NaN,演员
4,m51454521,27552814,沈婷婷,女,25727236,演员,是,7,25727236,201413,...,有评分,2.1,1413,1413.0,54.0,51454521,55105677,202523,NaN,演员
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1632,m53150931,27313518,伊能静,女,26575441,演员,是,4,26575441,情剑,...,有评分,2.8,1860,1860.0,72.0,53150931,54627085,549529,脱尘郡主,导演/演员
1633,m51665609,27573513,康帆,男,25832780,演员,是,3,25832780,武僧传奇之决战程子沟,...,有评分,2.8,243,243.0,26.0,51665609,55147075,131868,木瓜,演员
1634,m51665609,1353993,薛江涛,男,25832780,演员,是,4,25832780,武僧传奇之决战程子沟,...,有评分,2.8,243,243.0,26.0,51665609,2708035,144617,NaN,演员
1635,m51665609,27552599,阴海龙,男,25832780,演员,是,2,25832780,武僧传奇之决战程子沟,...,有评分,2.8,243,243.0,26.0,51665609,55105247,298814,NaN,演员


In [22]:
movie_net_data_top_min = get_chart_data_all(df_movie_top_n_min, title_prefix="低分电影关系网络")
# 保存处理后的数据
with open('json_data/movie_net_data_lowest.json', 'w', encoding='utf-8') as f:
    json.dump(movie_net_data_top_min, f, ensure_ascii=False, indent=4)

## TOP -N 1w

我从豆瓣中筛选出 评分人数超过1万的电影里，评分最低的200部作品。
其中最低只有 2.1 分，最高也不过 4.8 分。
使用这些作品的导演与主要演员数据，以“影人—电影”为连边关系，构建了一个无向图模型，进而得到了这批电影的合作关系网络。
为避免争议，不显示影人姓名。
友情提醒：超高清大图，双指可缩放，不是双击！不是双击！不是双击！
特别声明：
数据来源于公开的网络数据，旨在探索电影合作网络的结构特征，难免存在遗漏或错误，内容仅限学习与研究使用。

In [23]:
df_movie_rate_num_1w.sort_values(by='rating_num', ascending=True)

,movie_id_m,k_title,rating_num,k_rating_people
4462,m4710885,犬王,2.1,34475.0
4095,m52645597,纯洁心灵·逐梦演艺圈,2.2,97537.0
1045,m48327133,放手爱,2.3,25010.0
3516,m50601397,冰封侠：时空行者,2.7,11268.0
10738,m52832359,汽车人总动员,2.7,20423.0
...,...,...,...,...
10134,m2836087,大闹天宫,9.4,345809.0
8997,m2929101,横空出世,9.4,48528.0
983,m2922855,茶馆,9.5,72754.0
2158,m2615761,背靠背，脸对脸,9.5,74530.0


In [24]:
top_n = 200
df_movie_top_n = get_lowest_rated_movie_casts(df_movie_rate_num_1w,
                                              df_movie_raw,
                                              ascending=True,
                                              top_n=top_n)
df_movie_top_n

,movie_id_m,cast_id,cast_name,gender,movie_id,k_role,is_main_cast,index,id,k_title,...,is_rating,rating_num,rating_people,k_rating_people,rating_index,k_movie_id,k_cast_id,uid,portray,cast_role_agg
0,m4710885,27494952,海顿,男,2355418,演员,是,1,2355418,犬王,...,有评分,2.1,34475,34475.0,269.0,4710885,54989953,34849,NaN,导演/演员
1,m4710885,27543770,刘龙,男,2355418,演员,是,7,2355418,犬王,...,有评分,2.1,34475,34475.0,269.0,4710885,55087589,158228,NaN,演员
2,m4710885,27487838,高放,男,2355418,演员,是,4,2355418,犬王,...,有评分,2.1,34475,34475.0,269.0,4710885,54975725,221851,NaN,演员
3,m4710885,1367651,黄蕾蕾,男,2355418,演员,否,999,2355418,犬王,...,有评分,2.1,34475,34475.0,269.0,4710885,2735351,337177,NaN,演员
4,m4710885,27567426,王玉璋,男,2355418,演员,是,5,2355418,犬王,...,有评分,2.1,34475,34475.0,269.0,4710885,55134901,505119,NaN,演员
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3560,m66915483,27481097,王小利,男,33457717,演员,是,10,33457717,大红包,...,有评分,4.8,34993,34993.0,410.0,66915483,54962243,492757,假Ellie父亲,导演/演员
3561,m66915483,1422530,姜语心,女,33457717,演员,是,8,33457717,大红包,...,有评分,4.8,34993,34993.0,410.0,66915483,2845109,511004,小薇,演员
3562,m66915483,27480672,岳跃利,男,33457717,演员,是,12,33457717,大红包,...,有评分,4.8,34993,34993.0,410.0,66915483,54961393,541881,真Ellie父亲,演员
3563,m66915483,27560878,贾冰,男,33457717,演员,是,4,33457717,大红包,...,有评分,4.8,34993,34993.0,410.0,66915483,55121805,565456,钱好史,导演/演员


In [25]:
movie_net_data_top = get_chart_data_all(df_movie_top_n, title_prefix="低分电影关系网络")
# 保存处理后的数据
with open('json_data/movie_net_data_lowest_1w.json', 'w', encoding='utf-8') as f:
    json.dump(movie_net_data_top, f, ensure_ascii=False, indent=4)

In [26]:
df_movie_top_n['rating_num'].min(), df_movie_top_n['rating_num'].max()

(np.float64(2.1), np.float64(4.8))

In [27]:
top_desc = """
主流的电影研究往往集中在高分经典上——演员的奖项、导演的稳定创作力、以及票房表现等传统指标。
但我在想，既然好片能养出“黄金合作班底”，那这些经典佳作之间，是否也隐藏着某种合作网络？
于是，我从豆瓣中选取了
[向右R][向右R]评分人数超过10万人且评分最高的200部电影，
这批作品的分数高得离谱：最低都在 7.6分以上，最顶尖的甚至稳居影史榜单多年不动。
围绕这 200 部影片的 2700 位导演与主要演员（影人），我以“影人—电影”为连边关系，构建了一个无向图模型，进而得到了这批高分电影的合作关系网络，也就是下面的四张图。
[一R]图 1：
高分电影关系网（完整网络）
包含全部 200 部高分电影及其导演与主演。
可以看到整体结构相对庞大，包含多个连通子图。
有些子图像散落在影史角落的“小宇宙”，彼此之间并不直接关联。
如果你眼尖，说不定还能在图里找出某些你熟悉的组合。
[二R]图 2：
高分电影关系网（最大连通子图）
当我们剔除那些完全孤立的子图后，剩下的最大连通子图变得更加紧密，该子图包含175部电影及2574位影人。
不过仍然可以看到不少 “度为 1” 的影人节点——这意味着他们只在这 200 部电影中参与了一部作品。
为了让合作模式更清晰，我进一步剔除了这些度为 1 的节点，得到下方更集中的网络结构。
[三R]图 3：
高分电影关系网（去除度为1影人），此时剩余545位影人，而电影仍为200部。
可以看到部分电影节点变成孤立点，说明它们的主创人员并未与其他高分电影形成交叉合作。
但整体结构的形态开始显现，影史中那些频繁合作的黄金阵容也开始浮出水面。
[四R]图 4：
高分电影关系网（最大连通子图 + 去除度为1影人）
这是整个结构最紧凑、也最具“合作意义”的部分，此时剩余175部电影及539位影人。
影人之间的联系在这里最为密切，能看出哪些导演常与固定班底合作，以及哪些演员频繁在高质量项目中相遇。
研究这些高分电影的合作网络，能从另一种角度理解“经典是如何诞生的”：
不仅是导演与演员的个人实力，更是长期合作、默契搭档，以及稳定可靠的创作关系网在背后支撑。
当然，他们并不一定只拍佳作，高分电影只是他们职业轨迹上的亮眼部分。但正是这些高分作品，构成了他们在电影史中的共同坐标。
[向右R][向右R]特别声明：
数据来源于公开的网络数据，旨在探索电影合作网络的结构特征，难免存在遗漏或错误，内容仅限学习与研究使用。
#数据可视化 #经典电影推荐 #与电影对视120次 
"""

# 电视剧

In [49]:
df_tv.sort_values(by='rating_num', ascending=True)

,movie_id_m,k_title,rating_num,k_rating_people
4359,m7750069,秦可卿之谜,2.1,813.0
4277,m70321889,东八区的先生们,2.1,211287.0
5378,m52890803,K时代,2.2,1119.0
279,m21199455,敌后便衣队传奇,2.3,2690.0
7682,m10512369,新生活大爆炸,2.3,7096.0
...,...,...,...,...
1066,m7765479,武林外传,9.6,400757.0
1879,m3729669,红楼梦,9.7,168190.0
1118,m53207743,毛骗 终结篇,9.7,68590.0
2193,m4420051,大明王朝1566,9.7,145676.0


## TOP -N

In [54]:
top_n_min = 200
df_tv_top_n_min = get_lowest_rated_movie_casts(df_tv,
                                              df_tv_raw,
                                              ascending=True,
                                              top_n=top_n_min)
df_tv_top_n_min

,movie_id_m,cast_id,cast_name,gender,movie_id,k_role,is_main_cast,index,id,k_title,...,is_rating,rating_num,rating_people,k_rating_people,rating_index,k_movie_id,k_cast_id,uid,portray,cast_role_agg
0,m7750069,27482752,苗乙乙,女,3875010,演员,是,1,3875010,秦可卿之谜,...,有评分,2.1,813,813.0,41.0,7750069,54965553,36035,秦可卿、秦妃,演员
1,m7750069,27483159,博弘,女,3875010,演员,否,999,3875010,秦可卿之谜,...,有评分,2.1,813,813.0,41.0,7750069,54966367,81051,王熙凤,导演/演员
2,m7750069,27488205,陈剑飞,男,3875010,演员,是,3,3875010,秦可卿之谜,...,有评分,2.1,813,813.0,41.0,7750069,54976459,166393,NaN,导演/演员
3,m7750069,1317803,姚守岗,男,3875010,导演,是,1,3875010,秦可卿之谜,...,有评分,2.1,813,813.0,41.0,7750069,2635655,194191,NaN,导演
4,m7750069,1337502,麦文燕,女,3875010,演员,否,999,3875010,秦可卿之谜,...,有评分,2.1,813,813.0,41.0,7750069,2675053,349463,NaN,演员
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2866,m53297975,27249085,潘虹,女,26648963,演员,是,1,26648963,我们的爱,...,有评分,3.5,6742,6742.0,154.0,53297975,54498219,274543,齐舒兰,演员
2867,m53297975,27567703,陈牧扬,男,26648963,演员,是,7,26648963,我们的爱,...,有评分,3.5,6742,6742.0,154.0,53297975,55135455,322069,NaN,演员
2868,m53297975,27483460,朱亚英,女,26648963,演员,是,9,26648963,我们的爱,...,有评分,3.5,6742,6742.0,154.0,53297975,54966969,325284,许母,演员
2869,m53297975,1373881,杨锦斌,男,26648963,演员,否,999,26648963,我们的爱,...,有评分,3.5,6742,6742.0,154.0,53297975,2747811,365303,高尔夫球员,演员


In [63]:
tv_net_data_top_min = get_chart_data_all(df_tv_top_n_min, title_prefix="低分电视剧关系网络", work_type="电视剧")
# 保存处理后的数据
with open('json_data/tv_net_data_lowest.json', 'w', encoding='utf-8') as f:
    json.dump(tv_net_data_top_min, f, ensure_ascii=False, indent=4)

In [69]:
tv_info = df_tv_top_n_min[['k_movie_id', 'k_title', 'rating_num', 'k_movie_year', 'k_rating_people']].drop_duplicates().sort_values(by='rating_num').reset_index(drop=True)
tv_info['k_movie_id'] = tv_info['k_movie_id'].apply(lambda x: str(x))
tv_info.to_json('json_data/tv_info_lowest.json', orient='records', force_ascii=False, indent=4)

In [68]:
tv_info['rating_num'].min(), tv_info['rating_num'].max()

(np.float64(2.1), np.float64(3.5))

In [70]:
tv_info

,k_movie_id,k_title,rating_num,k_movie_year,k_rating_people
0,7750069,秦可卿之谜,2.1,1999,813.0
1,70321889,东八区的先生们,2.1,2022,211287.0
2,52890803,K时代,2.2,2010,1119.0
3,21199455,敌后便衣队传奇,2.3,2012,2690.0
4,10512369,新生活大爆炸,2.3,2010,7096.0
...,...,...,...,...,...
195,52470439,最美是你,3.5,2016,355.0
196,10774081,嫁到非洲,3.5,2000,502.0
197,52542199,黎明破晓前,3.5,2015,106.0
198,70741565,上有老下有小,3.5,2024,1218.0


## TOP +N 1W

In [87]:
top_n_max = 200
df_tv_top_n_max_5w = get_lowest_rated_movie_casts(df_tv_rate_num_5w,
                                              df_tv_raw,
                                              ascending=False,
                                              top_n=top_n_min)
df_tv_top_n_max_5w

,movie_id_m,cast_id,cast_name,gender,movie_id,k_role,is_main_cast,index,id,k_title,...,is_rating,rating_num,rating_people,k_rating_people,rating_index,k_movie_id,k_cast_id,uid,portray,cast_role_agg
0,m2883637,27483006,韩再芬,女,1441794,演员,否,999,1441794,走向共和,...,有评分,9.7,73275,73275.0,843.0,2883637,54966061,27494,沈玉英,导演/演员
1,m2883637,30414323,洪宗义,男,1441794,演员,否,999,1441794,走向共和,...,有评分,9.7,73275,73275.0,843.0,2883637,60828695,35241,方伯谦,演员
2,m2883637,27483118,马仑,男,1441794,演员,否,999,1441794,走向共和,...,有评分,9.7,73275,73275.0,843.0,2883637,54966285,58193,段祺瑞,演员
3,m2883637,27562254,刘伟明,男,1441794,演员,否,999,1441794,走向共和,...,有评分,9.7,73275,73275.0,843.0,2883637,55124557,65106,张謇,导演/演员
4,m2883637,27482432,赵立新,男,1441794,演员,否,999,1441794,走向共和,...,有评分,9.7,73275,73275.0,843.0,2883637,54964913,71288,罗文,演员
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8792,m3573529,27495105,陆诗雨,男,1786740,演员,是,14,1786740,还珠格格第二部,...,有评分,7.9,120844,120844.0,977.0,3573529,54990259,469107,柳青,导演/演员
8793,m3573529,27495266,李平,男,1786740,导演,是,1,1786740,还珠格格第二部,...,有评分,7.9,120844,120844.0,977.0,3573529,54990581,487123,NaN,导演/演员
8794,m3573529,27566488,孔庆三,男,1786740,演员,否,999,1786740,还珠格格第二部,...,有评分,7.9,120844,120844.0,977.0,3573529,55133025,531242,NaN,演员
8795,m3573529,27240644,李明启,女,1786740,演员,是,12,1786740,还珠格格第二部,...,有评分,7.9,120844,120844.0,977.0,3573529,54481337,543941,容嬷嬷,演员


In [88]:
tv_info_max_5w = df_tv_top_n_max_5w[['k_movie_id', 'k_title', 'rating_num', 'k_movie_year', 'k_rating_people']].drop_duplicates().sort_values(by='rating_num', ascending=False).reset_index(drop=True)
tv_info_max_5w['k_movie_id'] = tv_info_max_5w['k_movie_id'].apply(lambda x: str(x))
tv_info_max_5w.to_json('json_data/tv_info_lowest.json', orient='records', force_ascii=False, indent=4)
tv_info_max_5w

,k_movie_id,k_title,rating_num,k_movie_year,k_rating_people
0,2883637,走向共和,9.7,2003,73275.0
1,53207743,毛骗 终结篇,9.7,2015,68590.0
2,4420051,大明王朝1566,9.7,2007,145676.0
3,3729669,红楼梦,9.7,1987,168190.0
4,4313375,西游记,9.6,1986,236560.0
...,...,...,...,...,...
195,70930071,显微镜下的大明之丝绢案,7.9,2023,140158.0
196,52597919,鬼吹灯之精绝古城,7.9,2016,210691.0
197,71783665,欢颜,7.9,2023,68502.0
198,4086615,王子变青蛙,7.9,2005,68673.0


In [89]:
tv_net_data_top_max_5w = get_chart_data_all(df_tv_top_n_max_5w, title_prefix="高分电视剧关系网络", work_type="电视剧")
# 保存处理后的数据
with open('json_data/tv_net_data_top_200_max_5w.json', 'w', encoding='utf-8') as f:
    json.dump(tv_net_data_top_max_5w, f, ensure_ascii=False, indent=4)

In [90]:
tv_info_max_5w[tv_info_max_5w['k_title'] == '马大帅']

,k_movie_id,k_title,rating_num,k_movie_year,k_rating_people


# main

In [99]:
df_raw 

,cast_id,cast_name,gender,movie_id,k_role,is_main_cast,index,id,k_title,k_type,...,rating_num,rating_people,k_rating_people,rating_index,k_movie_id,k_cast_id,uid,portray,movie_id_m,cast_role_agg
0,1017182,陈燕燕,女,10771202,演员,是,1,10771202,深闺疑云,电影,...,7.8,92,92.0,27.0,21542453,2034413,54,赵兰,m21542453,演员
1,1043845,刘志荣,男,10581318,演员,是,11,10581318,四大家族之龙虎兄弟,电影,...,7.8,2290,2290.0,134.0,21162685,2087739,75,NaN,m21162685,导演/演员
2,1050373,茅瑛,女,10573512,演员,是,1,10573512,鬼娘子,电影,...,5.8,235,235.0,37.0,21147073,2100795,77,NaN,m21147073,演员
3,1124360,利智,女,10581318,演员,是,3,10581318,四大家族之龙虎兄弟,电影,...,7.8,2290,2290.0,134.0,21162685,2248769,92,NaN,m21162685,演员
4,1314471,王挺,男,10531190,演员,是,1,10531190,惊天动地,电视剧,...,8.5,326,326.0,53.0,21062429,2628991,146,NaN,m21062429,导演/演员
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
256797,1354509,林泳淘,女,26984991,演员,否,999,26984991,婚姻合伙人,电视剧,...,5.8,1259,1259.0,85.0,53970031,2709067,604125,邓秀玉,m53970031,演员
256798,27496534,刘芊蒂,女,2337597,演员,是,2,2337597,操行零分,电影,...,7.0,211,211.0,38.0,4675243,54993117,604128,NaN,m4675243,演员
256799,27565771,董亚春,女,2270516,导演,是,2,2270516,八月一日,电影,...,6.4,1123,1123.0,85.0,4541081,55131591,604130,NaN,m4541081,导演
256800,35415094,张可,女,7053738,演员,否,999,7053738,樱桃,电视剧,...,5.7,2003,2003.0,107.0,14107525,70830237,604135,NaN,m14107525,演员


In [106]:
work_type = '电视剧'
work_type_value = 'movie' if work_type == '电影' else 'tv'
sort_by = 'rating_num'
ascending = False
asc_value = 'min' if ascending else 'max'
title_value = '低分' if ascending else '高分'
top_n = 200
rating_people_min = 30000
df_filtered = works_filter(df_raw, work_type=work_type, sort_by=sort_by, ascending=ascending, top_n=top_n, rating_people_min=rating_people_min)
df_filtered

,movie_id_m,cast_id,cast_name,gender,movie_id,k_role,is_main_cast,index,id,k_title,...,is_rating,rating_num,rating_people,k_rating_people,rating_index,k_movie_id,k_cast_id,uid,portray,cast_role_agg
0,m2883637,27483006,韩再芬,女,1441794,演员,否,999,1441794,走向共和,...,有评分,9.7,73275,73275.0,843.0,2883637,54966061,27494,沈玉英,导演/演员
1,m2883637,30414323,洪宗义,男,1441794,演员,否,999,1441794,走向共和,...,有评分,9.7,73275,73275.0,843.0,2883637,60828695,35241,方伯谦,演员
2,m2883637,27483118,马仑,男,1441794,演员,否,999,1441794,走向共和,...,有评分,9.7,73275,73275.0,843.0,2883637,54966285,58193,段祺瑞,演员
3,m2883637,27562254,刘伟明,男,1441794,演员,否,999,1441794,走向共和,...,有评分,9.7,73275,73275.0,843.0,2883637,55124557,65106,张謇,导演/演员
4,m2883637,27482432,赵立新,男,1441794,演员,否,999,1441794,走向共和,...,有评分,9.7,73275,73275.0,843.0,2883637,54964913,71288,罗文,演员
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8509,m70470905,27480802,赵滨,男,35235428,演员,否,999,35235428,对手,...,有评分,8.1,32779,32779.0,515.0,70470905,54961653,540174,陈华,演员
8510,m70470905,27581615,何蓝逗,女,35235428,演员,是,7,35235428,对手,...,有评分,8.1,32779,32779.0,515.0,70470905,55163279,550854,李小满,演员
8511,m70470905,35439368,从瑞麟,男,35235428,演员,否,999,35235428,对手,...,有评分,8.1,32779,32779.0,515.0,70470905,70878785,566366,徐志良,演员
8512,m70470905,27482871,房子斌,男,35235428,演员,否,999,35235428,对手,...,有评分,8.1,32779,32779.0,515.0,70470905,54965791,567598,老怼,演员


In [107]:
df_info = df_filtered[['k_movie_id', 'k_title', 'rating_num', 'k_movie_year', 'k_rating_people']].drop_duplicates().sort_values(by='rating_num', ascending=ascending).reset_index(drop=True)
df_info['k_movie_id'] = df_info['k_movie_id'].apply(lambda x: str(x))
df_info.to_json(f'json_data/works_info_{work_type_value}_{sort_by}_{asc_value}_{top_n}_{rating_people_min}.json', orient='records', force_ascii=False, indent=4)
df_info

,k_movie_id,k_title,rating_num,k_movie_year,k_rating_people
0,2883637,走向共和,9.7,2003,73275.0
1,4420051,大明王朝1566,9.7,2007,145676.0
2,3729669,红楼梦,9.7,1987,168190.0
3,53207743,毛骗 终结篇,9.7,2015,68590.0
4,4313375,西游记,9.6,1986,236560.0
...,...,...,...,...,...
195,4313457,包青天,8.2,1993,48443.0
196,55328045,隐秘而伟大,8.1,2020,172830.0
197,4309535,封神榜,8.1,1990,42295.0
198,53707693,异人之下,8.1,2023,108957.0


In [ ]:
net_data = get_chart_data_all(df_filtered, title_prefix=f"{title_value}{work_type}关系网络", work_type=work_type)
# 保存处理后的数据
with open(f'json_data/net_data_{work_type_value}_{sort_by}_{asc_value}_{top_n}_{rating_people_min}.json', 'w', encoding='utf-8') as f:
    json.dump(net_data, f, ensure_ascii=False, indent=4)

# 数据统计

In [30]:
top_n = 200
df_movie_top_n = get_lowest_rated_movie_casts(df_movie_rate_num_10w,
                                              df_movie_raw,
                                              ascending=False,
                                              top_n=top_n)
df_movie_top_n

,movie_id_m,cast_id,cast_name,gender,movie_id,k_role,is_main_cast,index,id,k_title,...,is_rating,rating_num,rating_people,k_rating_people,rating_index,k_movie_id,k_cast_id,uid,portray,cast_role_agg
0,m2583141,1003494,张国荣,男,1291546,演员,是,1,1291546,霸王别姬,...,有评分,9.6,1783862,1783862.0,4138.0,2583141,2007037,389,程蝶衣(小豆子,导演/演员
1,m2583141,27226213,蒋雯丽,女,1291546,演员,是,6,1291546,霸王别姬,...,有评分,9.6,1783862,1783862.0,4138.0,2583141,54452475,1784,小豆子生母,导演/演员
2,m2583141,27256028,张丰毅,男,1291546,演员,是,2,1291546,霸王别姬,...,有评分,9.6,1783862,1783862.0,4138.0,2583141,54512105,3236,段小楼(小石头,演员
3,m2583141,27206595,葛优,男,1291546,演员,是,4,1291546,霸王别姬,...,有评分,9.6,1783862,1783862.0,4138.0,2583141,54413239,3384,袁世卿(袁四爷,演员
4,m2583141,1275231,李春,男,1291546,演员,是,14,1291546,霸王别姬,...,有评分,9.6,1783862,1783862.0,4138.0,2583141,2550511,8887,少年小四,演员
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3985,m43883657,1458583,李彩霞,女,21941804,演员,是,8,21941804,白日焰火,...,有评分,7.6,362165,362165.0,1659.0,43883657,2917215,454389,NaN,演员
3986,m43883657,35422753,王英涛,女,21941804,演员,否,999,21941804,白日焰火,...,有评分,7.6,362165,362165.0,1659.0,43883657,70845555,460080,居委会吕主任,演员
3987,m43883657,27480917,王景春,男,21941804,演员,是,4,21941804,白日焰火,...,有评分,7.6,362165,362165.0,1659.0,43883657,54961883,462373,荣荣,演员
3988,m43883657,1375912,彭龙,男,21941804,演员,是,12,21941804,白日焰火,...,有评分,7.6,362165,362165.0,1659.0,43883657,2751873,533987,便衣,导演/演员


In [31]:
df_movie_top_n_asc = get_lowest_rated_movie_casts(df_movie_rate_num_1w,
                                              df_movie_raw,
                                              ascending=True,
                                              top_n=top_n)
df_movie_top_n_asc

,movie_id_m,cast_id,cast_name,gender,movie_id,k_role,is_main_cast,index,id,k_title,...,is_rating,rating_num,rating_people,k_rating_people,rating_index,k_movie_id,k_cast_id,uid,portray,cast_role_agg
0,m4710885,27494952,海顿,男,2355418,演员,是,1,2355418,犬王,...,有评分,2.1,34475,34475.0,269.0,4710885,54989953,34849,NaN,导演/演员
1,m4710885,27543770,刘龙,男,2355418,演员,是,7,2355418,犬王,...,有评分,2.1,34475,34475.0,269.0,4710885,55087589,158228,NaN,演员
2,m4710885,27487838,高放,男,2355418,演员,是,4,2355418,犬王,...,有评分,2.1,34475,34475.0,269.0,4710885,54975725,221851,NaN,演员
3,m4710885,1367651,黄蕾蕾,男,2355418,演员,否,999,2355418,犬王,...,有评分,2.1,34475,34475.0,269.0,4710885,2735351,337177,NaN,演员
4,m4710885,27567426,王玉璋,男,2355418,演员,是,5,2355418,犬王,...,有评分,2.1,34475,34475.0,269.0,4710885,55134901,505119,NaN,演员
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3560,m66915483,27481097,王小利,男,33457717,演员,是,10,33457717,大红包,...,有评分,4.8,34993,34993.0,410.0,66915483,54962243,492757,假Ellie父亲,导演/演员
3561,m66915483,1422530,姜语心,女,33457717,演员,是,8,33457717,大红包,...,有评分,4.8,34993,34993.0,410.0,66915483,2845109,511004,小薇,演员
3562,m66915483,27480672,岳跃利,男,33457717,演员,是,12,33457717,大红包,...,有评分,4.8,34993,34993.0,410.0,66915483,54961393,541881,真Ellie父亲,演员
3563,m66915483,27560878,贾冰,男,33457717,演员,是,4,33457717,大红包,...,有评分,4.8,34993,34993.0,410.0,66915483,55121805,565456,钱好史,导演/演员


In [32]:
# 计数
def get_df_stats(df):
    # 作品数量
    works_count = df['movie_id_m'].nunique()
    # 导演数量
    director_count = df[df['k_role'] == '导演']['k_cast_id'].nunique()
    # 演员数量
    actor_count = df[df['k_role'] == '演员']['k_cast_id'].nunique()
    # 影人数量
    cast_count = df['k_cast_id'].nunique()
    # 最高分
    rating_max = df['rating_num'].max()
    # 最低分
    rating_min = df['rating_num'].min()
    # 平均分
    rating_mean = df['rating_num'].mean()
    return {
        'works_count': works_count,
        'director_count': director_count,
        'actor_count': actor_count,
        'cast_count': cast_count,
        'rating_max': rating_max,
        'rating_min': rating_min,
        'rating_mean': round(rating_mean, 2),
        'rating_people_num': '10W+'
    }

In [33]:
# 图指标
def get_graph_metrics(G, df, is_bipartite=False):
    metrics = {}
    # 度
    degrees = dict(G.degree())
    # 节点数
    metrics['num_nodes'] = G.number_of_nodes()
    # 边数
    metrics['num_edges'] = G.number_of_edges()
    # 介数中心性最大值节点
    betweenness = nx.betweenness_centrality(G)

    if is_bipartite:
        # 最大度
        max_degree = max(degrees.values())
        metrics['max_degree'] = max_degree
        if max_degree > 1:
            # 最大度节点
            max_degree_node = [n for n in G.nodes() if degrees[n] == max_degree]
            # metrics['max_degree_node'] = max_degree_node
            # 最大度节点名称
            max_degree_node_names = []
            for n in max_degree_node:
                if 'm' in str(n):
                    name = df[df['movie_id_m'] == n]['k_title'].unique().tolist()
                else:
                    name = df[df['k_cast_id'] == n]['cast_name'].unique().tolist()
                max_degree_node_names.extend(name)
            metrics['max_degree_node_names'] = ",".join(max_degree_node_names)
        # 最大介数中心性
        max_betweenness = max(betweenness.values())
        metrics['max_betweenness'] = round(max_betweenness, 6)
        if max_betweenness > 0:
            # 最大介数中心性节点
            max_betweenness_node = [n for n in G.nodes() if betweenness[n] == max_betweenness]
            # metrics['max_betweenness_node'] = max_betweenness_node
            # 最大介数中心性节点名称
            max_betweenness_node_names = []
            for n in max_betweenness_node:
                if 'm' in str(n):
                    name = df[df['movie_id_m'] == n]['k_title'].unique().tolist()
                else:
                    name = df[df['k_cast_id'] == n]['cast_name'].unique().tolist()
                max_betweenness_node_names.extend(name)
            metrics['max_betweenness_node_names'] = ",".join(max_betweenness_node_names)
    else:
        # 影人节点
        cast_nodes = [n for n in G.nodes() if 'm' not in str(n)]
        # 影人节点数
        metrics['num_cast_nodes'] = len(cast_nodes)
        # 影人节点最大度
        max_cast_degree = max([degrees[n] for n in cast_nodes])
        metrics['max_cast_degree'] = max_cast_degree
        if max_cast_degree > 1:
            # 最大度影人节点
            max_cast_degree_node = [n for n in cast_nodes if degrees[n] == max_cast_degree]
            # 最大度影人节点名称
            max_cast_degree_node_names = df[df['k_cast_id'].isin(max_cast_degree_node)]['cast_name'].unique().tolist()
            # metrics['max_cast_degree_node'] = max_cast_degree_node
            metrics['max_cast_degree_node_names'] = ",".join(max_cast_degree_node_names)
        # 介数中心性最大值影人节点
        max_cast_betweenness = max([betweenness[n] for n in cast_nodes])
        metrics['max_cast_betweenness'] = round(max_cast_betweenness, 6)
        if max_cast_betweenness > 0:
            # 最大介数中心性影人节点
            max_cast_betweenness_node = [n for n in cast_nodes if betweenness[n] == max_cast_betweenness]
            # 最大介数中心性影人节点名称
            max_cast_betweenness_node_names = df[df['k_cast_id'].isin(max_cast_betweenness_node)]['cast_name'].unique().tolist()
            # metrics['max_cast_betweenness_node'] = max_cast_betweenness_node
            metrics['max_cast_betweenness_node_names'] = ",".join(max_cast_betweenness_node_names)

        # 电影节点
        movie_nodes = [n for n in G.nodes() if 'm' in str(n)]
        # 电影节点数
        metrics['num_movie_nodes'] = len(movie_nodes)
        # 电影节点最大度
        max_movie_degree = max([degrees[n] for n in movie_nodes])
        metrics['max_movie_degree'] = max_movie_degree
        if max_movie_degree > 1:
            # 最大度电影节点
            max_movie_degree_node = [n for n in movie_nodes if degrees[n] == max_movie_degree]
            # 最大度电影节点名称
            max_movie_degree_node_names = df[df['movie_id_m'].isin(max_movie_degree_node)]['k_title'].unique().tolist()
            # metrics['max_movie_degree_node'] = max_movie_degree_node
            metrics['max_movie_degree_node_names'] = ",".join(max_movie_degree_node_names)
        # 最大介数中心性电影节点
        max_movie_betweenness = max([betweenness[n] for n in movie_nodes])
        metrics['max_movie_betweenness'] = round(max_movie_betweenness, 6)
        if max_movie_betweenness > 0:
            # 最大介数中心性电影节点
            max_movie_betweenness_node = [n for n in movie_nodes if betweenness[n] == max_movie_betweenness]
            # 最大介数中心性电影节点名称
            max_movie_betweenness_node_names = df[df['movie_id_m'].isin(max_movie_betweenness_node)]['k_title'].unique().tolist()
            # metrics['max_movie_betweenness_node'] = max_movie_betweenness_node
            metrics['max_movie_betweenness_node_names'] = ",".join(max_movie_betweenness_node_names)

    # 密度
    metrics['density'] = round(nx.density(G), 6)
    # 平均聚类系数
    # metrics['average_clustering'] = nx.average_clustering(G)
    # 是否连通图
    metrics['is_connected'] = "是" if nx.is_connected(G) else "否"
    # 直径（仅适用于连通图）
    if nx.is_connected(G):
        metrics['diameter'] = nx.diameter(G)
    else:
        # metrics['diameter'] = None
        # 子图数量
        metrics['num_connected_components'] = nx.number_connected_components(G)
        # 最大子图
        largest_cc = max(nx.connected_components(G), key=len)
        G_largest = G.subgraph(largest_cc).copy()
        metrics['largest_cc_num_nodes'] = G_largest.number_of_nodes()
        metrics['largest_cc_num_nodes_ratio'] = f"{round(G_largest.number_of_nodes() / G.number_of_nodes() * 100, 2)}%"
        metrics['largest_cc_num_edges'] = G_largest.number_of_edges()
        metrics['largest_cc_num_edges_ratio'] = f"{round(G_largest.number_of_edges() / G.number_of_edges() * 100, 2)}%"

    return metrics

In [34]:
# main
def movie_net_stats(df):
    G = get_G(df)
    stats = {}
    stats['df_stats'] = get_df_stats(df)
    stats['graph_metrics'] = get_graph_metrics(G, df)
    if stats['graph_metrics']['is_connected'] == "否":
        largest_cc = max(nx.connected_components(G), key=len)
        stats['largest_connected_component'] = get_graph_metrics(G.subgraph(largest_cc).copy(), df)
    # 二分图-电影
    movie_nodes = [n for n in G.nodes() if 'm' in str(n)]
    G_movie = bipartite.projected_graph(G, movie_nodes)
    stats['movie_bipartite_graph_metrics'] = get_graph_metrics(G_movie, df, is_bipartite=True)
    # 最大连通子图二分图-电影
    if not nx.is_connected(G_movie):
        largest_cc = max(nx.connected_components(G_movie), key=len)
        stats['movie_bipartite_largest_connected_component'] = get_graph_metrics(G_movie.subgraph(largest_cc).copy(), df, is_bipartite=True)
    # 二分图-影人
    cast_nodes = [n for n in G.nodes() if 'm' not in str(n)]
    G_cast = bipartite.projected_graph(G, cast_nodes)
    stats['cast_bipartite_graph_metrics'] = get_graph_metrics(G_cast, df, is_bipartite=True)
    # 最大连通子图二分图-影人
    if not nx.is_connected(G_cast):
        largest_cc = max(nx.connected_components(G_cast), key=len)
        stats['cast_bipartite_largest_connected_component'] = get_graph_metrics(G_cast.subgraph(largest_cc).copy(), df, is_bipartite=True)
    return stats


In [35]:
top_stats = movie_net_stats(df_movie_top_n)

In [36]:
top_stats_asc = movie_net_stats(df_movie_top_n_asc)

In [44]:
# 中英文对照字典
key_to_chinese = {
    'works_count': '作品数量',
    'director_count': '导演数量',
    'actor_count': '演员数量',
    'cast_count': '影人数量',
    'rating_max': '最高评分',
    'rating_min': '最低评分',
    'rating_mean': '平均评分',
    'rating_people_num': '评分人数标准',
    'num_nodes': '节点数',
    'num_edges': '边数',
    'max_degree': '最大度',
    'max_degree_node': '最大度节点',
    'max_degree_node_names': '最大度节点名称',
    'max_betweenness': '最大介数中心性',
    'max_betweenness_node': '最大介数中心性节点',
    'max_betweenness_node_names': '最大介数中心性节点名称',
    'num_cast_nodes': '影人节点数',
    'max_cast_degree': '影人最大度',
    'max_cast_degree_node': '影人最大度节点',
    'max_cast_degree_node_names': '影人最大度节点名称',
    'max_cast_betweenness': '影人最大介数中心性',
    'max_cast_betweenness_node': '影人最大介数中心性节点',
    'max_cast_betweenness_node_names': '影人最大介数中心性节点名称',
    'num_movie_nodes': '作品节点数',
    'max_movie_degree': '作品最大度',
    'max_movie_degree_node': '作品最大度节点',
    'max_movie_degree_node_names': '作品最大度节点名称',
    'max_movie_betweenness': '作品最大介数中心性',
    'max_movie_betweenness_node': '作品最大介数中心性节点',
    'max_movie_betweenness_node_names': '作品最大介数中心性节点名称',
    'density': '密度',
    'average_clustering': '平均聚类系数',
    'is_connected': '是否连通',
    'diameter': '直径',
    'num_connected_components': '连通子图数量',
    'largest_cc_num_nodes': '最大连通子图节点数',
    'largest_cc_num_nodes_ratio': '最大连通子图节点比例',
    'largest_cc_num_edges': '最大连通子图边数',
    'largest_cc_num_edges_ratio': '最大连通子图边比例'
}
# 指标说明
metric_description = {
    'works_count': '电影/电视剧数量',
    'director_count': '导演总数',
    'actor_count': '演员总数', 
    'cast_count': '所有影人（包括导演和演员）的总数',
    'rating_max': '所有作品中的最高评分值',
    'rating_min': '所有作品中的最低评分值',
    'rating_mean': '所有作品评分的平均值',
    'rating_people_num': '为保证数据质量，选取评分人数不少于该标准的作品，两者标准可能不同',
    'num_nodes': '网络中节点的总数量',
    'num_edges': '网络中边的总数量',
    'max_degree': '节点中最大的度值（度：节点连接的边数，反映直接影响力）',
    'max_degree_node': '具有最大度值的节点ID',
    'max_degree_node_names': '具有最大度值的节点名称',
    'max_betweenness': '节点中最大的介数中心性值（介数中心性：衡量节点作为桥梁的重要性）',
    'max_betweenness_node': '具有最大介数中心性的节点ID',
    'max_betweenness_node_names': '具有最大介数中心性的节点名称',
    'num_cast_nodes': '影人类型节点的数量',
    'max_cast_degree': '影人节点中最大的度值（度：节点连接的边数，这里是影人参与的作品数量）',
    'max_cast_degree_node': '具有最大度值的影人节点ID',
    'max_cast_degree_node_names': '具有最大度值的影人姓名',
    'max_cast_betweenness': '影人节点中最大的介数中心性值（介数中心性：衡量节点作为桥梁的重要性）',
    'max_cast_betweenness_node': '具有最大介数中心性的影人节点ID',
    'max_cast_betweenness_node_names': '具有最大介数中心性的影人姓名',
    'num_movie_nodes': '作品类型节点的数量',
    'max_movie_degree': '作品节点中最大的度值（度：节点连接的边数，反映合作影人数量）',
    'max_movie_degree_node': '具有最大度值的作品节点ID',
    'max_movie_degree_node_names': '具有最大度值的作品名称',
    'max_movie_betweenness': '作品节点中最大的介数中心性值（介数中心性：衡量节点在网络连通中的关键程度）',
    'max_movie_betweenness_node': '具有最大介数中心性的作品节点ID',
    'max_movie_betweenness_node_names': '具有最大介数中心性的作品名称',
    'density': '网络的密度，表示实际边数与可能最大边数的比值',
    'average_clustering': '网络中所有节点的聚类系数的平均值',
    'is_connected': '判断网络是否连通（即任意两个节点间都存在路径）',
    'diameter': '网络的直径，即所有节点对之间最短路径的最大长度',
    'num_connected_components': '网络中连通子图的数量',
    'largest_cc_num_nodes': '网络中最大连通子图包含的节点数量',
    'largest_cc_num_nodes_ratio': '最大连通子图节点数占网络总节点数的比例',
    'largest_cc_num_edges': '网络中最大连通子图包含的边数量',
    'largest_cc_num_edges_ratio': '最大连通子图边数占网络总边数的比例'
}
stats_parts_dict = {
    'df_stats': '原始数据统计',
    'graph_metrics': '整体网络图指标',
    'largest_connected_component': '最大连通子图指标',
    'movie_bipartite_graph_metrics': '作品二分图指标',
    'movie_bipartite_largest_connected_component': '作品二分图最大连通子图指标',
    'cast_bipartite_graph_metrics': '影人二分图指标',
    'cast_bipartite_largest_connected_component': '影人二分图最大连通子图指标'
}

In [46]:
# 将两个字典中的指标，按部分合并，之后加入excel的sheet中
# writer = pd.ExcelWriter('movie_network_stats.xlsx', engine='xlsxwriter')
with pd.ExcelWriter('movie_network_stats.xlsx', engine='xlsxwriter') as writer:
    for p in stats_parts_dict.keys():
        rows = []
        for key in top_stats[p].keys():
            row = {}
            row['指标'] = key_to_chinese.get(key, key)
            row['高分电影网络'] = str(top_stats[p][key])
            row['低分电影网络'] = str(top_stats_asc[p][key])
            row['说明'] = metric_description.get(key, '')
            rows.append(row)
        df_part = pd.DataFrame(rows)
        sheet_name = stats_parts_dict[p]
        print(sheet_name)
        df_part.to_excel(writer, sheet_name=sheet_name, index=False)
    # df_part.to_excel(f'movie_network_stats.xlsx', sheet_name=sheet_name, index=False)
    # df_part.to_excel(writer, sheet_name=sheet_name, index=False)


原始数据统计
整体网络图指标
最大连通子图指标
作品二分图指标
作品二分图最大连通子图指标
影人二分图指标
影人二分图最大连通子图指标


In [39]:
pd.DataFrame.from_dict(top_stats_asc['graph_metrics'], orient='index', columns=['值']).rename(index=key_to_chinese)

,值
节点数,2613
边数,3543
影人节点数,2413
影人最大度,15
影人最大度节点名称,包贝尔
影人最大介数中心性,0.045633
影人最大介数中心性节点名称,包贝尔
电影节点数,200
电影最大度,45
电影最大度节点名称,三少爷的剑


In [40]:
# 片名转为csv文件
def df_movie_titles_to_csv(df, filename, asc=False):
    movie_list = df.sort_values(by='rating_num', ascending=asc)['k_title'].unique()
    # 转为csv, 10列，每列20个电影
    movie_df = pd.DataFrame()
    num_cols = 5
    num_rows = int(np.ceil(len(movie_list) / num_cols))
    for i in range(num_cols):
        col_data = movie_list[i * num_rows:(i + 1) * num_rows]
        movie_df[f'{i*40}-{(i+1)*40}'] = pd.Series(col_data)
    movie_df.to_csv(filename, index=False)

In [41]:
df_movie_titles_to_csv(df_movie_top_n, 'movie_top_200.csv')
df_movie_titles_to_csv(df_movie_top_n_asc, 'movie_lowest_200.csv', asc=True)

简单介绍二分图的概念：
二分图是一种将网络节点划分为两个互不相交的集合（如影人集合和电影集合），且所有边只在不同集合节点之间连接的图结构。
具体来说：在影人-电影二分图中，影人节点之间不相连，电影节点之间也不相连，所有的合作关系都通过影人与电影之间的边来表示，这种结构完美地建模了"谁参演了什么电影"的关联关系。
基于二分图投影：我们可以将这种二分网络拆分为两个单模网络——影人合作网络（通过共同参演电影建立连接）和电影关联网络（通过共享影人建立连接），从而分别分析影人间的合作模式和电影间的题材关联。

[向右R][向右R]特别声明：
数据来源于公开的网络数据，旨在探索电影合作网络的结构特征，难免存在遗漏或错误，仅供娱乐，请勿过分解读。
